[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/fluids/smoke_rings/smoke_rings.ipynb)

# Smoke Rings

This example moves thin vortex filaments, such as the core of a smoke ring, through an ideal fluid, both by their own bending and by each other's pull. A filament is modelled as a closed polygon of vorticity in a fluid without viscosity that cannot be compressed. By the Biot–Savart law every part of a filament moves the fluid everywhere, and the fluid carries every other part along. The part from close to each point dominates and moves the point across the curve, perpendicular to the curve and to the way it bends, at a speed set by how sharply it bends. Keeping that part alone is called the smoke ring flow, or the local induction approximation; under it a circle flies along its axis without changing shape, and the surface a curve sweeps over time is called its Hashimoto surface, after Hoffmann's discrete flow for polygons of unit edges (2000). Keeping the whole law lets two rings, one behind the other, take turns passing through each other.

The filament's points and edges are intrinsic antivectors, and its circulation is an intrinsic scalar. The turn between the two edges at a vertex is the rotor `outgoing * ~incoming`, and the edges' commutator over the turn's scalar part plus the product of their lengths is the tangent of half the turning angle in the turn's plane; times the circulation it is a point, the vertex's step along the binormal, and on unit edges exactly Hoffmann's velocity. The polygon's vector area, the sum of `point & next_point` over its edges, is the filament's impulse, and the flow keeps it. Hoffmann's Bäcklund transformation steps the flow in discrete time by turning each offset about a quad's diagonal with a quaternion, `((edge - offset) + twist).normalized() >> offset`, and the whole Biot–Savart law sums `separations.commutator(segments)` over every segment of every filament.

In Gibbs–Heaviside vector notation, the Biot–Savart law $\mathbf u(\mathbf x) = \frac{\Gamma}{4\pi}\oint \frac{d\boldsymbol\ell \times (\mathbf x - \mathbf y)}{|\mathbf x - \mathbf y|^3}$ reduces near the filament to $\dot{\boldsymbol\gamma} = \frac{\Gamma}{4\pi}\ln\frac{R}{a}\,\kappa\,\mathbf T \times \mathbf N$, with $\mathbf T = \boldsymbol\gamma'$ and $\mathbf N = \mathbf T'/|\mathbf T'|$ in arc length, $R$ the radius of curvature and $a$ the core radius, and on a polygon of unit edges $\mathbf S_k$ Hoffmann's flow reads $\dot{\boldsymbol\gamma}_k = 2\,\mathbf S_{k-1} \times \mathbf S_k / (1 + \mathbf S_{k-1} \cdot \mathbf S_k)$.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ https://github.com/EelcoHoogendoorn/numga/archive/HEAD.tar.gz"

In [ ]:
%matplotlib inline
from collections.abc import Callable, Iterator
from itertools import islice

import numpy as np

from numga import NumpyContext, concatenate, stack
from numga.algebras import VGA3D as ga
from examples.util.animation import save_figure, show_animation
from examples.fluids.smoke_rings import render
from examples.fluids.smoke_rings.scenarios import coaxial_rings, ring, torus_knot, wavy_ring

mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()                                # Scalar+
Vector = ga.gatype.vector()                                # Vector+
Rotor = ga.gatype.rotor()                                  # Rotor+
# Points, and their differences, the edges and offsets, are antivectors, intrinsic.
Point = ga.gatype.antivector()                             # Antivector-
# The filament's circulation changes sign under a mirror, an intrinsic scalar.
Circulation = ga.gatype.antipseudoscalar()                 # Antipseudoscalar-
# The Bäcklund transformation's parameter, added to the quad's diagonal, a point, is an intrinsic scalar too.
Parameter = ga.gatype.antipseudoscalar()                   # Antipseudoscalar-
# A polygon's vector area, the join of its points, is an intrinsic vector, a patch.
Patch = ga.gatype.antibivector()                           # Antibivector-

## The turn at a vertex

At a vertex of a polygon, the outgoing edge times the reverse of the incoming one, `outgoing * ~incoming`, is the turn from one edge to the other. Its bivector over its scalar part plus the product of the edge lengths is the tangent of half the turning angle in the turn's plane, and times the circulation, an intrinsic scalar, that bivector is a point, the step along the binormal at the vertex. Twice the step over the mean length of the two edges is the vertex's velocity, exactly Hoffmann's on edges of unit length and the curvature times the binormal in the limit of short edges. A regular polygon makes the same turn at every vertex, so the ring flies along its axis without changing shape, at twice the tangent of half its turn per vertex.

In [ ]:
# A regular polygon's vertices, its edges of unit length; the circulation, one in the flow's rescaled time.
n_ring = 60
circulation = mv(Circulation, [1.0])                       # [] Circulation-

In [ ]:
def edges(points: Point) -> Point:
    """The edges of closed polygons, edge k running from point k to point k + 1."""
    closed_polyline = concatenate([points, points[..., :1]], axis=-1)            # [...] Point-
    return closed_polyline[..., 1:] - closed_polyline[..., :-1]                  # [...] Point-


def velocity(points: Point, circulation: Circulation) -> Point:
    """The smoke ring flow's velocity at each vertex of closed polygonal filaments of the given circulation, twice the
    Cayley parameter of the turn at the vertex times the circulation, over the mean length of its two edges."""
    # The polygon closed at both ends, so that every vertex has an incoming and an outgoing side.
    wrapped = concatenate([points[..., -1:], points, points[..., :1]], axis=-1)  # [...] Point-
    sides = wrapped[..., 1:] - wrapped[..., :-1]                                 # [...] Point-
    incoming, outgoing = sides[..., :-1], sides[..., 1:]                         # [...] Point-, Point-
    lengths = outgoing.norm() * incoming.norm()                                  # [...] Scalar+
    # The commutator of the two edges is the bivector of the turn; over the lengths plus the edges paired,
    # it is the tangent of half the turning angle.
    binormal_step = 2 * incoming.commutator(outgoing) * circulation / (lengths + outgoing.dot(incoming))   # [...] Point-
    return binormal_step / ((outgoing.norm() + incoming.norm()) / 2)             # [...] Point-


ring_velocities = velocity(ring(n_ring), circulation)                            # [n_ring] Point-

In [ ]:
print("speed along the axis, and twice the tangent of the half turn:", (ring_velocities & mv.z).abs().to_array()[0], 2 * np.tan(np.pi / n_ring))

## The flow's invariants and the surface it sweeps

Each vertex moves perpendicular to both of its edges, so no edge changes its length, whatever the lengths are. The flow keeps the polygon's vector area too, the sum of each point joined with the next, which is the impulse of a vortex filament. A ring carrying a standing wave of three crests flies along its axis while the waves travel around it. Joined by strips from one time to the next, its curves make the surface it sweeps, its Hashimoto surface.

In [ ]:
# The wavy ring's vertices, the waves around it and the tilt of its edges; the flow's step, its number
# of steps, and the steps between kept rings; the duration of a frame.
n_wavy, wavy_mode, wavy_tilt = 90, 3, 0.4
surface_step, n_surface_steps, surface_stride = 0.4, 750, 10
frame_duration_ms = 40

In [ ]:
def impulse(points: Point) -> Patch:
    """The vector area of closed polygons, twice over, the sum of each point joined with the next."""
    return (points & (points + edges(points))).sum(axis=-1)                      # [...] Patch-


def flow(points: Point, rate: Callable[[Point], Point], step: float) -> Iterator[Point]:
    """Yield the polygons of a flow with the given velocity at every classical Runge–Kutta step, without end."""
    while True:
        yield points
        first = rate(points)
        second = rate(points + first * (step / 2))
        third = rate(points + second * (step / 2))
        fourth = rate(points + third * step)
        points = points + (first + 2 * second + 2 * third + fourth) * (step / 6)


def smoke_ring_velocity(points: Point) -> Point:
    """The smoke ring flow's velocity at every vertex."""
    return velocity(points, circulation)                                         # [...] Point-


wavy = wavy_ring(n_wavy, wavy_mode, wavy_tilt)                                   # [n_wavy] Point-
surface = stack(list(islice(flow(wavy, smoke_ring_velocity, surface_step), 0, n_surface_steps + 1, surface_stride)))   # [...] Point-
lengths = edges(surface).norm()                                                  # [...] Scalar+
areas = impulse(surface)                                                         # [...] Patch-

In [ ]:
save_figure(render.draw_surface(surface), "smoke_rings_surface");

The surface is coloured from early, sand, to late, blue. The animation shows the same ring flying, each part coloured by its speed, from blue at rest to green at the fastest.

In [ ]:
show_animation(render.animate_curves(surface, smoke_ring_velocity(surface).norm()), "smoke_rings_wavy", frame_duration_ms)

## A knotted vortex

A vortex filament tied in a trefoil knot on a torus about the axis turns about the axis as it flies along it, and keeps its knotted shape while its loops ripple, because the flow moves each part of the knot by its own bending and the trefoil's three loops bend alike. The knot's edges are shorter than one and unequal, so its step is shorter, and its velocity, divided by the mean edge length, is the curvature times the binormal in the limit of short edges.

In [ ]:
# The trefoil's windings around the axis and around the tube, the tube's radii and its vertices; the
# flow's step, its number of steps, and the steps between frames.
knot_windings, knot_radii, n_knot = (2, 3), (6.0, 2.5), 240
knot_step, n_knot_steps, knot_stride = 1 / 30, 3600, 40

In [ ]:
knot = torus_knot(*knot_windings, *knot_radii, n_knot)                          # [n_knot] Point-
curves = stack(list(islice(flow(knot, smoke_ring_velocity, knot_step), 0, n_knot_steps + 1, knot_stride)))   # [...] Point-
knot_lengths = edges(curves).norm()                                             # [...] Scalar+
knot_areas = impulse(curves)                                                    # [...] Patch-

In [ ]:
show_animation(render.animate_curves(curves, smoke_ring_velocity(curves).norm()), "smoke_rings_knot", frame_duration_ms)

## Discrete time

Hoffmann's Bäcklund transformation makes time discrete as well. The new polygon is the old one moved by an offset of one fixed length at every vertex, with every new edge again of unit length. Each offset follows from the one before by a fold about the diagonal of the quad between the old edge and the new one, `edge - offset`, a point. Plus the transformation's parameter, an intrinsic scalar like the circulation, the diagonal is a quaternion, and the fold is its sandwich, `((edge - offset) + twist).normalized() >> offset`, which keeps each new edge's part along the diagonal, so every new edge is exactly as long as the old.

For a closed polygon the offsets must come round to where they started. On the sphere of unit offsets, a fold across an edge acts as a Lorentz transformation, a boost along the edge and a turn about it, so a lap around the polygon is a product of rotors, and minus the last edge, carried once around, lands on the lap's attracting fixed point. With unit offsets the new polygon is the old one moved by the smoke ring flow over about half the parameter and relabelled by one vertex, so a transformation is a step in time in which every edge keeps exactly unit length and the ring, a discrete elastic curve, moves exactly rigidly. The wavy ring stepped this way sweeps the discrete Hashimoto surface, every quad folded about its diagonal.

In quaternion notation, with the vectors as imaginary quaternions, the fold is the Riccati-type equation $(\lambda + \tilde S)(\lambda + \operatorname{Re}\nu + v) = (\lambda + \operatorname{Re}\nu + v_{+})(\lambda + S)$ for all $\lambda$, with $\tilde S$ the new edge and $v$, $v_{+}$ the offsets at its ends, and the Lorentz transformation the matrix of $SL(2, \mathbb{C})$ that acts on the sphere by the Möbius map.

In [ ]:
# The transformation's parameter, an intrinsic scalar; the rings' vertices, and the steps.
twist = mv(Parameter, [0.5])                                                     # [] Parameter-
n_discrete, n_discrete_steps = 40, 100

In [ ]:
def next_offset(edge: Point, offset: Point, twist: Parameter) -> Point:
    """The next offset of a Bäcklund transformation across an edge, the current one turned about the diagonal
    `edge - offset` of the quad between them by the quaternion the diagonal and the parameter make together."""
    return ((edge - offset) + twist).normalized() >> offset                      # [...] Point-


def backlund(points: Point, start: Point, twist: Parameter) -> Point:
    """The Bäcklund transformation of open polygons with the given parameter and first offset, every later offset
    the one before folded across the edge between them."""
    sides = points[..., 1:] - points[..., :-1]                                   # [...] Point-

    # The offsets are folded one after another; a product of the folds' Lorentz rotors grows from edge to edge until
    # round-off swamps the offsets.
    def offsets() -> Iterator[Point]:
        offset = start
        yield offset
        for index in range(sides.shape[-1]):
            offset = next_offset(sides[..., index], offset, twist)
            yield offset

    return points + stack(list(offsets()), axis=-1)                              # [...] Point-

In [ ]:
def periodic_offset(points: Point, twist: Parameter) -> Point:
    """The unit offset at the first vertex of closed polygons of unit edges that comes round to itself after a lap,
    minus the last edge carried once around by the lap's Lorentz transformation onto its attracting fixed point."""
    sides = edges(points)                                                        # [...] Point-
    # Every fold is a boost along its edge and a turn about it, its rapidity and the turn's angle the real and
    # imaginary parts of the logarithm of 1 - 2 i / twist.
    rapidity = (1 + 4 / twist.squared()).log() / 2                               # [] Scalar+
    angle = -(2 * twist.inverse()).arctan()                                      # [] Parameter-
    folds = (-(rapidity * sides.dual_inverse() + angle * sides) / 2).exp()       # [...] Multivector+
    lap = folds.cumprod(axis=-1)[..., -1]                                        # [...] Multivector+
    # A Lorentz transformation acts on paravectors by the sandwich with the reverse; the offset's null paravector is
    # one plus the direction it stands for.
    image = lap * (1 - sides[..., -1].dual_inverse()) * lap.reverse()            # [...] Multivector+
    # The paravector's bivector and volume parts are zero.
    return (image.cast(Vector) / image.cast(Scalar)).dual()                      # [...] Point-


def time_step(points: Point, twist: Parameter) -> Point:
    """The closed polygons of unit edges one step of the discrete-time smoke ring flow later, by the Bäcklund
    transformation with periodic offsets relabelled by one vertex, a time of about half the parameter."""
    closed_polyline = concatenate([points, points[..., :1]], axis=-1)            # [...] Point-
    return backlund(closed_polyline, periodic_offset(points, twist), twist)[..., 1:]   # [...] Point-


# The fold, and the Lorentz transformation of the same edge, on a few unit offsets.
edge = mv.z.dual()                                                               # [] Point-
unit_offsets = mv.vector([[1, 0, 0], [0, 1, 0], [0, -0.6, 0.8]]).dual()          # [3] Point-
rapidity, angle = (1 + 4 / twist.squared()).log() / 2, -(2 * twist.inverse()).arctan()   # [] Scalar+, Parameter-
rotor = (-(rapidity * edge.dual_inverse() + angle * edge) / 2).exp()            # [] Multivector+
image = rotor * (1 + unit_offsets.dual_inverse()) * rotor.reverse()              # [3] Multivector+
lorentz_offsets = (image.cast(Vector) / image.cast(Scalar)).dual()               # [3] Point-

# The plain ring and the wavy one, stepped together.
rings = stack([ring(n_discrete), wavy_ring(n_discrete, wavy_mode, wavy_tilt)])   # [2, n_discrete] Point-


def discrete_steps() -> Iterator[Point]:
    """Yield the rings at every step of the discrete-time flow."""
    polygons = rings
    while True:
        yield polygons
        polygons = time_step(polygons, twist)


discrete = stack(list(islice(discrete_steps(), n_discrete_steps + 1)))           # [n_discrete_steps + 1, 2, n_discrete] Point-
final_ring = discrete[-1, 0]                                                     # [n_discrete] Point-

In [ ]:
save_figure(render.draw_surface(discrete[:, 1]), "smoke_rings_discrete");

In [ ]:
show_animation(render.animate_curves(discrete, smoke_ring_velocity(discrete).norm()), "smoke_rings_discrete_flow", frame_duration_ms)

## A loop running along a straight vortex

A soliton is a solitary wave that travels without changing its shape. On a straight vortex line it is a loop that runs along the line and turns about it as it goes, Hasimoto's soliton, and one Bäcklund transformation of a straight line makes one. An offset leaning just off the forward direction stays near it for a while, then crosses over to the backward one, and the new polygon carries a loop where it crosses. The offset's length sets the loop's size, and the parameter how drawn out it is along the line.

The steps of discrete time carry the loop along, the line running on straight past its ends. Through the loop the tangent turns once around, and the angles of the turns at the vertices, `outgoing * ~incoming`, add up to a full turn at every step, whatever the loop's size.

In Frenet–Serret notation, Hasimoto's map $\psi = \kappa\, e^{i\int\tau\, ds}$ of the curvature $\kappa$ and the torsion $\tau$ takes the smoke ring flow to the nonlinear Schrödinger equation $i\psi_t + \psi_{ss} + \tfrac{1}{2}|\psi|^2\psi = 0$, and the soliton's curvature is $\kappa = 2\nu\operatorname{sech}(\nu s)$, whose integral is $2\pi$ for every $\nu$.

In [ ]:
# A straight line of unit edges; how far the first offset leans off the forward direction, its length and
# the parameter; the parameter of the steps that carry the loop, and their number, one a frame.
n_line, lean, offset_length = 75, 1e-5, 4.0
loop_twist, carry_twist = mv(Parameter, [2.0]), mv(Parameter, [1.0])           # [] Parameter-, Parameter-
n_carry_steps = 90

In [ ]:
def turns(points: Point) -> Rotor:
    """The turn at every inner vertex of polygons of unit edges, the rotor from the incoming edge to the outgoing
    one, the square of the rotor that carries one onto the other."""
    sides = points[..., 1:] - points[..., :-1]                                  # [...] Point-
    return sides[..., 1:] * ~sides[..., :-1]                                    # [...] Rotor+


line = mv.x.dual() * (np.arange(n_line) - n_line / 2)                           # [n_line] Point-
start = ((mv.zx * lean).exp() >> mv.x.dual()) * offset_length                   # [] Point-


def soliton_steps() -> Iterator[Point]:
    """Yield the line with its loop at every step of the discrete-time flow."""
    curve = backlund(line, start, loop_twist)                                   # [n_line] Point-
    while True:
        yield curve
        # Offsets from minus the first edge, relabelled by one vertex; the line runs on straight past its end.
        relabelled = backlund(curve, curve[0] - curve[1], carry_twist)[1:]      # [n_line - 1] Point-
        curve = concatenate([relabelled, 2 * relabelled[-1:] - relabelled[-2:-1]])   # [n_line] Point-


solitons = stack(list(islice(soliton_steps(), n_carry_steps + 1)))              # [n_carry_steps + 1, n_line] Point-
angles = turns(solitons).log().norm()                                           # [n_carry_steps + 1, n_line - 2] Scalar+
# The flow's speed along the line, and at its two ends, which the closing edge would bend, their neighbours'.
inner = smoke_ring_velocity(solitons).norm()[..., 1:-1]                         # [n_carry_steps + 1, n_line - 2] Scalar+
line_speeds = concatenate([inner[..., :1], inner, inner[..., -1:]], axis=-1)    # [n_carry_steps + 1, n_line] Scalar+

In [ ]:
print("total turning through the loop at the first and the last step, in full turns:", (angles[[0, -1]].sum(axis=-1) / (2 * np.pi)).to_array())
show_animation(render.animate_trail(solitons, line_speeds), "smoke_rings_soliton", frame_duration_ms)

In [ ]:
save_figure(render.draw_turns(turns(solitons)), "smoke_rings_turns");

The animation draws the line at each step and, behind it, the screw-shaped tube the loop sweeps. The figure draws the turning angles along the line and through the steps, a streak of one width moving at one speed.

## Two rings leapfrogging

The whole Biot–Savart law moves every vertex by every segment of every filament, the other filaments' included. A straight segment moves a point by the commutator of the point's separation from the segment's middle with the segment, over the cube of the separation's length, softened by a core radius so that the law stays finite on the filament itself; this is the vortex filament of Rosenhead and Moore. Two equal rings, one a little behind the other on the same axis, move each other. The front ring widens and slows, the rear one narrows, speeds up and passes through the front one, and the two change places, twice in every cycle.

In Gibbs–Heaviside vector notation, a polygonal filament of circulation $\Gamma$ and core radius $\delta$ induces $\mathbf u(\mathbf x) = \frac{\Gamma}{4\pi}\sum_j \frac{\mathbf S_j \times (\mathbf x - \mathbf m_j)}{(|\mathbf x - \mathbf m_j|^2 + \delta^2)^{3/2}}$ at a point $\mathbf x$, with $\mathbf m_j$ the middle of the segment $\mathbf S_j$.

In [ ]:
# Two equal rings on one axis, one behind the other, their radius, vertices and heights, their circulations
# and core radius; the flow's step and number of steps, through two cycles, and the steps between frames.
radius, n_frog, frog_heights = 5.0, 48, np.array([0.0, 2.5])
circulations, core_radius = mv(Circulation, [[10.0], [10.0]]), 0.4             # [2] Circulation-, float
frog_step, n_frog_steps, frog_stride = 0.4 / 3, 150, 2

In [ ]:
def induced_velocity(points: Point, circulations: Circulation, core_radius: float) -> Point:
    """The velocity at every vertex of closed polygonal vortex filaments of the given circulations, induced by every
    segment of every filament, the commutator of the vertex's separation from the segment's middle with the segment
    over the softened cube of the separation."""
    n_rings, n_vertices = points.shape
    segments = edges(points)                                                     # [n_rings, n_vertices] Point-
    separations = points.reshape(-1)[:, None] - (points + segments / 2).reshape(-1)[None, :]   # [n_rings * n_vertices, n_rings * n_vertices] Point-
    soft_squares = separations.norm_squared() + core_radius**2                   # [n_rings * n_vertices, n_rings * n_vertices] Scalar+
    strengths = circulations[..., None].broadcast_to(points.shape).reshape(-1) / (4 * np.pi)   # [n_rings * n_vertices] Circulation-
    pulls = separations.commutator(segments.reshape(-1)) * strengths / (soft_squares * soft_squares.square_root())
    return pulls.sum(axis=1).reshape(points.shape)                               # [n_rings, n_vertices] Point-


def leapfrog_velocity(points: Point) -> Point:
    """The velocity the leapfrogging rings induce at every vertex of both."""
    return induced_velocity(points, circulations, core_radius)                   # [2, n_frog] Point-


frog_rings = coaxial_rings(radius, n_frog, frog_heights)                         # [2, n_frog] Point-
frogs = stack(list(islice(flow(frog_rings, leapfrog_velocity, frog_step), n_frog_steps + 1)))   # [n_frog_steps + 1, 2, n_frog] Point-
# Each point paired with the axis is its height along it.
heights = (frogs & mv.z).mean(axis=-1)                                           # [n_frog_steps + 1, 2] Scalar+
radii = (frogs - mv.z.dual() * (frogs & mv.z)).norm().mean(axis=-1)              # [n_frog_steps + 1, 2] Scalar+
# The steps just before the rings change places.
ahead = heights[:, 1] > heights[:, 0]
passes = np.flatnonzero(ahead[1:] != ahead[:-1])

In [ ]:
print("times at which one ring passes through the other:", passes * frog_step)
save_figure(render.draw_swept(frogs), "smoke_rings_swept");

The figure draws the tube each ring sweeps through two cycles in its own colour, with the half nearer the viewer cut away. Wherever one ring passes through the other, its tube runs inside.

In [ ]:
# Each time's rings pushed by both; the induced flow couples every vertex of a time, and times are a batch.
frog_speeds = stack([leapfrog_velocity(frame) for frame in frogs]).norm()        # [n_frog_steps + 1, 2, n_frog] Scalar+
show_animation(render.animate_curves(frogs[::frog_stride], frog_speeds[::frog_stride]), "smoke_rings_leapfrog",
               frame_duration_ms)

In [ ]:
# checks
# The ring flies rigidly along its axis at twice the tangent of half its turn per vertex.
assert ((ring_velocities - ring_velocities[0]).norm() < 1e-13).all()
assert (((ring_velocities & mv.z).abs() - 2 * np.tan(np.pi / n_ring)).abs() < 1e-12).all()
# Both flows keep every edge's length and the polygon's vector area; the tolerance on the area tests the time steps.
assert ((lengths - lengths[0]).abs() < 1e-9).all()
assert ((knot_lengths - knot_lengths[0]).abs() < 1e-9).all()
assert ((areas - areas[0]).norm() < 1e-6 * areas[0].norm()).all()
assert ((knot_areas - knot_areas[0]).norm() < 1e-3 * knot_areas[0].norm()).all()
# The fold is the Lorentz transformation of its edge; in discrete time every edge keeps unit length, and the ring
# moves rigidly, all its points keeping one height.
assert ((lorentz_offsets - next_offset(edge, unit_offsets, twist)).norm() < 1e-12).all()
assert ((edges(discrete).norm() - 1).abs() < 1e-12).all()
assert (((final_ring - final_ring[0]) & mv.z).abs() < 1e-12).all()
# The soliton keeps unit edges, and the tangent turns once around through the loop at every step; the
# tolerance allows for the loop's tails beyond the line's ends.
assert (((solitons[:, 1:] - solitons[:, :-1]).norm() - 1).abs() < 1e-12).all()
assert ((angles.sum(axis=-1) - 2 * np.pi).abs() < 1e-3).all()
# The leapfrogging rings take turns, the narrower one passing through the wider.
assert len(passes) >= 4 and (radii[passes].argmin(axis=-1) == heights[passes].argmin(axis=-1)).all()